# Reproducción de fidelidad: Hur et al. (2022), Ansatz 8, MNIST

**Objetivo**: validar que el ansatz 8 (`U_6`) de Hur, Kim & Park (2022), *"Quantum
convolutional neural network for classical data classification"*, reproduce con
fidelidad razonable la exactitud publicada (98.7% ± 0.1%) bajo el protocolo de
entrenamiento de este benchmark.

Toda la lógica reutilizable (carga de datos, muestreo estratificado, representación
PCA, circuito, ciclo de entrenamiento, métricas) vive en `src/qcnn_benchmark/` —
este notebook solo configura y ejecuta. Ver `notebooks/dev/01_reproduce_hur.ipynb`
para la versión exploratoria original (con la bitácora completa de las dos
correcciones de encargo resueltas durante su desarrollo: par de clases del
objetivo 98.7% ± 0.1%, e identidad del circuito "Ansatz 8" = `U_6`) y
`src/qcnn_benchmark/models/qcnn_hur.py` para la atribución de licencia
(Apache-2.0, `external/hur_qcnn/`, https://github.com/takh04/QCNN) y el
detalle del circuito.

Este notebook corre **ambos pares de clases**: 0-vs-1 (el par que efectivamente
reporta 98.7% ± 0.1% en la Tabla I del paper -- validación de fidelidad) y 1-vs-8
(reportado como referencia adicional, sin comparación contra 98.7% ya que Hur et
al. no publican ese número para ese par).

**Nota sobre datasets (actualizada 2026-08-24)**: 1-vs-8 fue el dataset asignado
por el diseño de benchmark original (`context/diseño_experimentos (2).pdf`,
14 ago 2026). Ese diseño quedó **obsoleto** tras el rediseño de Semana 2
(24 ago 2026), que lo reemplaza por tres datasets nuevos para E0A en adelante:
Fashion-MNIST coat vs. shirt, MNIST 4 vs. 9 y MNIST 1 vs. 0. La corrida 1-vs-8
de este notebook se conserva como referencia histórica, no como parte del
protocolo vigente. La corrida **0-vs-1** (fidelidad contra Hur et al.) sí sigue
siendo válida tal cual -- y de hecho usa exactamente la misma partición de
MNIST que el nuevo dataset "1 vs 0" de E0A (`stratified_split` sobre las
mismas dos clases produce la misma partición train/val/test
independientemente de cuál dígito se marque como positivo; solo cambian las
etiquetas y, por construcción, no la exactitud alcanzable).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

from qcnn_benchmark.data import load_mnist_pool
from qcnn_benchmark.representations import build_pca_dataset
from qcnn_benchmark.models import qcnn_hur
from qcnn_benchmark.training import train_binary_classifier, uniform_pi_init
from qcnn_benchmark.metrics import batch_accuracy, predict_labels

print("Parámetros entrenables (Ansatz 8):", qcnn_hur.TOTAL_PARAMS, "(Tabla I del paper: 36)")


## 1. Carga de datos y representación (PCA densa, 16 componentes, [0, π])

Muestreo estratificado balanceado: 500 entrenamiento / 250 validación / 500
prueba por clase, semilla `20260802`. El PCA (16 componentes) y el escalado a
[0, π] se ajustan **solo** sobre el conjunto de entrenamiento (ver
`qcnn_benchmark.representations.pca`).


In [ ]:
X_ALL, Y_ALL = load_mnist_pool()
print("Pool MNIST combinado (train+test oficiales):", X_ALL.shape, Y_ALL.shape)

print("0 vs 1:")
rep_0v1 = build_pca_dataset(X_ALL, Y_ALL, class_pos=1, class_neg=0)

print("1 vs 8:")
rep_1v8 = build_pca_dataset(X_ALL, Y_ALL, class_pos=1, class_neg=8)


## 2. Entrenamiento

Protocolo exacto (`qcnn_benchmark.training.train_binary_classifier`): BCE, Adam
(lr=0.01, β1=0.9, β2=0.999), 200 actualizaciones, lote de 25, recorte de norma
global de gradiente a 5.0, early stopping (paciencia = 5 chequeos de validación,
δ=1e-4, chequeo cada 10 actualizaciones), inicialización uniforme en [-π, π],
selección de checkpoint por menor pérdida de validación.

**Semillas: 1 (`RUN_SEED = 0`).** El protocolo formal de este framework usa
5 semillas por configuración (media ± desviación estándar, ver diseño de
experimentos). Aquí se usa 1 semilla a propósito porque este notebook es la
**validación de fidelidad del adaptador** (¿el circuito `U_6` reproduce el
número publicado por Hur et al.?), no el experimento estadístico E0A -- ese
notebook sí correrá las 5 semillas y reportará media ± desviación.


In [ ]:
RUN_SEED = 0


def plot_loss_curve(result, title):
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(range(1, result["n_updates_run"] + 1), result["train_loss_history"],
            label="Pérdida de entrenamiento (por actualización)", alpha=0.7)
    val_updates, val_losses = zip(*result["val_loss_history"])
    ax.plot(val_updates, val_losses, "o-", label="Pérdida de validación (cada 10 actualizaciones)", color="darkorange")
    if result["stopped_early_at"]:
        ax.axvline(result["stopped_early_at"], color="red", linestyle="--", alpha=0.6, label="Early stopping")
    ax.set_xlabel("Actualización de parámetros")
    ax.set_ylabel("Entropía cruzada binaria")
    ax.set_title(title)
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_confusion(y_true, y_pred, title, class_labels):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    fig, ax = plt.subplots(figsize=(4.5, 4))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks([0, 1]); ax.set_xticklabels(class_labels)
    ax.set_yticks([0, 1]); ax.set_yticklabels(class_labels)
    ax.set_xlabel("Predicción"); ax.set_ylabel("Etiqueta real")
    ax.set_title(title)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black")
    plt.colorbar(im, ax=ax, fraction=0.046)
    plt.tight_layout()
    plt.show()
    print(cm)
    return cm


def report_accuracy(train_acc, test_acc, tag, target_mean=None, target_std=None):
    print("=" * 70)
    print(f"{tag} -- Ansatz 8 (Hur et al.), 1 semilla de ejecución")
    print("=" * 70)
    print(f"Exactitud de entrenamiento: {train_acc * 100:.2f}%")
    print(f"Exactitud de prueba:        {test_acc * 100:.2f}%")
    if target_mean is None:
        print("Sin objetivo publicado por Hur et al. para este par de clases.")
        return
    gap_pp = test_acc * 100 - target_mean
    print(f"Objetivo publicado (Tabla I, 5 repeticiones): {target_mean}% ± {target_std}%")
    print(f"Diferencia (prueba - objetivo): {gap_pp:+.2f} puntos porcentuales")
    if abs(gap_pp) <= 1.0:
        print("-> Dentro de ~1pp del objetivo publicado: fidelidad razonable para 1 semilla.")
    elif abs(gap_pp) <= 3.0:
        print("-> Brecha moderada (1-3pp). Aceptable para 1 semilla vs. una media de 5 repeticiones.")
    else:
        print("-> Brecha grande (>3pp). Revisar antes de escalar a 5 semillas.")


## 3. Corrida A -- MNIST 0 vs 1 (validación de fidelidad contra 98.7% ± 0.1%)

In [ ]:
result_0v1 = train_binary_classifier(
    qcnn_hur.predict_proba, qcnn_hur.TOTAL_PARAMS, rep_0v1, uniform_pi_init,
    run_seed=RUN_SEED, tag="0v1",
)
plot_loss_curve(result_0v1, "MNIST 0 vs 1 -- Ansatz 8 (Hur et al.) -- curva de pérdida")


In [ ]:
params_0v1 = result_0v1["params"]
train_acc_0v1 = batch_accuracy(qcnn_hur.predict_proba, params_0v1, rep_0v1["X_train"], rep_0v1["y_train"])
test_acc_0v1 = batch_accuracy(qcnn_hur.predict_proba, params_0v1, rep_0v1["X_test"], rep_0v1["y_test"])
report_accuracy(train_acc_0v1, test_acc_0v1, "MNIST 0 vs 1", target_mean=98.7, target_std=0.1)


In [ ]:
y_pred_test_0v1 = predict_labels(qcnn_hur.predict_proba, params_0v1, rep_0v1["X_test"])
_ = plot_confusion(rep_0v1["y_test"], y_pred_test_0v1,
                    "Matriz de confusión -- MNIST 0 vs 1 (prueba)",
                    ["0 (dígito 0)", "1 (dígito 1)"])


## 4. Corrida B -- MNIST 1 vs 8 (referencia histórica, dataset obsoleto para E0A)

**Sin comparación contra 98.7% ± 0.1%**: ese número es el resultado publicado por
Hur et al. para 0 vs 1, no para 1 vs 8. Esta corrida se reporta como referencia
adicional, no como validación de fidelidad. 1-vs-8 era el dataset asignado por
el diseño de benchmark del 14-ago-2026; el rediseño de Semana 2 (24-ago-2026) lo
reemplazó -- ya no forma parte del protocolo de E0A en adelante (ver nota en la
Sec. 0).


In [ ]:
result_1v8 = train_binary_classifier(
    qcnn_hur.predict_proba, qcnn_hur.TOTAL_PARAMS, rep_1v8, uniform_pi_init,
    run_seed=RUN_SEED, tag="1v8",
)
plot_loss_curve(result_1v8, "MNIST 1 vs 8 -- Ansatz 8 (Hur et al.) -- curva de pérdida")


In [ ]:
params_1v8 = result_1v8["params"]
train_acc_1v8 = batch_accuracy(qcnn_hur.predict_proba, params_1v8, rep_1v8["X_train"], rep_1v8["y_train"])
test_acc_1v8 = batch_accuracy(qcnn_hur.predict_proba, params_1v8, rep_1v8["X_test"], rep_1v8["y_test"])
report_accuracy(train_acc_1v8, test_acc_1v8, "MNIST 1 vs 8")


In [ ]:
y_pred_test_1v8 = predict_labels(qcnn_hur.predict_proba, params_1v8, rep_1v8["X_test"])
_ = plot_confusion(rep_1v8["y_test"], y_pred_test_1v8,
                    "Matriz de confusión -- MNIST 1 vs 8 (prueba)",
                    ["0 (dígito 8)", "1 (dígito 1)"])


## 5. Resumen

| Par de clases | Exactitud entrenamiento | Exactitud prueba | Objetivo Hur et al. | Comparable |
|---|---|---|---|---|
| 0 vs 1 | ver celda de resultados | ver celda de resultados | 98.7% ± 0.1% (Tabla I, 5 rep.) | Sí |
| 1 vs 8 | ver celda de resultados | ver celda de resultados | No publicado | No |

**Próximos pasos**: E0A corre sobre los tres datasets del rediseño de Semana 2
(Fashion coat vs. shirt, MNIST 4 vs. 9, MNIST 1 vs. 0), no sobre 0-vs-1 ni
1-vs-8 directamente -- aunque, como se nota en la Sec. 0, la partición de
MNIST de "1 vs 0" coincide exactamente con la de "0 vs 1" usada aquí. Este
notebook sigue siendo solo la validación de fidelidad del adaptador contra el
número publicado por Hur et al., no el experimento E0A en sí.
